In [41]:
# Cell 1 — Install dependencies
!pip install -q pandas scikit-learn xgboost openai matplotlib seaborn plotly python-dotenv


In [42]:
# Cell 2 — Imports & helpers
import os, json, textwrap
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# helper: display small dataframe nicely
from IPython.display import display, HTML
display(HTML("<style>table {font-size: 12px}</style>"))


In [43]:
# Cell 3 — Create large sample resume dataset (120 candidates)

import pandas as pd
import random

names = [
    "Aarav", "Vivaan", "Aditya", "Vihaan", "Arjun", "Reyansh", "Sai", "Krishna", "Ishaan", "Rohan",
    "Ananya", "Diya", "Sara", "Isha", "Aditi", "Pooja", "Sanya", "Nisha", "Riya", "Meera",
    "Raj", "Kiran", "Neha", "Akash", "Sneha", "Rahul", "Priya", "Suresh", "Anjali", "Vikram",
    "Deepa", "Sameer", "Aishwarya", "Ravi", "Lakshmi", "Aman", "Manish", "Divya", "Karthik", "Gauri",
    "Dev", "Chitra", "Abhinav", "Varun", "Rekha", "Nikhil", "Tanya", "Farah", "Pavan", "Hina",
    "Gaurav", "Rachit", "Anmol", "Harini", "Simran", "Asha", "Yash", "Shweta", "Kabir", "Ankit",
    "Naveen", "Charu", "Esha", "Payal", "Sunil", "Aarohi", "Harsha", "Jaya", "Ritu", "Dinesh",
    "Poonam", "Mukesh", "Rohit", "Amrita", "Meenal", "Tarun", "Soham", "Alisha", "Mohan", "Nandini",
    "Kavya", "Pratik", "Rupal", "Rakesh", "Tina", "Usha", "Suman", "Hemant", "Chirag", "Aparna",
    "Bhavana", "Sridhar", "Lalit", "Vinay", "Monika", "Tanvi", "Kushal", "Dipti", "Parth", "Pritam",
    "Ajay", "Gagan", "Arpita", "Jay", "Mitali", "Omkar", "Piyush", "Nikita", "Raman", "Ameer"
]

skills_pool = [
    "python", "sql", "pandas", "numpy", "aws", "azure", "gcp", "spark", "hadoop", "hive",
    "kafka", "airflow", "java", "spring", "scala", "docker", "kubernetes", "terraform", "linux",
    "powerbi", "tableau", "ml", "deep-learning", "pytorch", "tensorflow", "xgboost", "scikit-learn",
    "react", "node", "javascript", "html", "css", "flask", "django", "fastapi", "devops",
    "data-engineering", "etl", "analytics", "leadership", "communication", "product-management",
    "cloud-security", "testing", "git", "agile", "rest-api"
]

educations = ["B.Tech", "M.Tech", "MBA", "B.Sc", "MCA", "BCA", "B.E"]
roles = ["Data Engineer", "ML Engineer", "Data Analyst", "Software Developer", "DevOps Engineer", "Product Manager", "Backend Developer", "Frontend Developer"]

# Function to create realistic skill combinations
def random_skills():
    return ",".join(random.sample(skills_pool, random.randint(4, 7)))

data = []
for i in range(1, 121):
    name = random.choice(names) + " " + random.choice(["Kumar", "Sharma", "Iyer", "Patel", "Roy", "Nair", "Singh", "Mehta", "Ali", "Das"])
    experience = random.randint(1, 12)
    edu = random.choice(educations)
    skills = random_skills()
    role = random.choice(roles)

    data.append({
        "candidate_id": i,
        "name": name,
        "skills": skills,
        "experience": experience,
        "education": edu,
        "role": role
    })

df = pd.DataFrame(data)

# Save as CSV
df.to_csv("sample_resumes.csv", index=False)
print("✅ Saved sample_resumes.csv with", len(df), "entries")

# Show preview
display(df.head(10))


✅ Saved sample_resumes.csv with 120 entries


,candidate_id,name,skills,experience,education,role
0,1,Rohit Das,"leadership,javascript,tableau,css",2,M.Tech,DevOps Engineer
1,2,Nisha Ali,"terraform,docker,agile,kafka,kubernetes,produc...",10,MCA,Data Analyst
2,3,Aishwarya Iyer,"aws,hadoop,scala,react,git",1,MCA,Backend Developer
3,4,Deepa Kumar,"flask,java,django,testing",9,B.Tech,Data Analyst
4,5,Ajay Das,"javascript,gcp,analytics,powerbi",11,MBA,Backend Developer
5,6,Aditya Ali,"communication,spring,terraform,product-managem...",5,B.Sc,Product Manager
6,7,Raman Singh,"git,numpy,tensorflow,data-engineering,cloud-se...",4,MCA,DevOps Engineer
7,8,Payal Das,"ml,fastapi,linux,django,product-management",12,MCA,Software Developer
8,9,Aman Mehta,"gcp,analytics,data-engineering,terraform,docke...",1,B.Sc,Product Manager
9,10,Lalit Mehta,"airflow,terraform,hadoop,html,spark",12,B.Tech,Software Developer


In [44]:
# Cell 4 — Optional: set your OpenAI key here (or skip to fallback summarizer)
# If you want to use OpenAI, set OPENAI_API_KEY below, e.g.:
# os.environ["OPENAI_API_KEY"] = "sk-xxxx"
# Uncomment & set if you have a key.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"


In [45]:
# Cell 5 — Resume summarizer (two modes: OpenAI or fallback)
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))  # if env key present, use OpenAI

if USE_OPENAI:
    import openai
    openai.api_key = os.getenv("OPENAI_API_KEY")
    print("OpenAI key detected — using OpenAI for summarization.")
else:
    print("No OpenAI key found. Using fallback rule-based summarizer.")

def fallback_summarize(text, max_bullets=3):
    # Very simple rule-based summarizer: split skills, experience, education
    parts = []
    # try to find skills (comma-separated in our sample)
    if isinstance(text, str) and "," in text:
        skills = text
        parts.append("Skills: " + skills)
    return " | ".join(parts)

def openai_summarize_resume(resume_text):
    # Minimal prompt — adapt for better outputs
    resp = openai.ChatCompletion.create(
        model="gpt-4o-mini" if "gpt-4o-mini" in openai.Model.list()['data'][0]['id'] else "gpt-4o-mini",
        messages=[{"role":"user","content":f"Summarize this candidate's profile in 3 short bullets:\n\n{resume_text}"}],
        max_tokens=150,
        temperature=0.2
    )
    return resp['choices'][0]['message']['content'].strip()

# Apply summarization to dataset (for demo we summarize the skills string)
summaries = []
for idx, row in df.iterrows():
    text = f"Name: {row['name']}\nSkills: {row['skills']}\nExperience: {row['experience']} years\nEducation: {row['education']}"
    if USE_OPENAI:
        try:
            s = openai_summarize_resume(text)
        except Exception as e:
            print("OpenAI call failed:", e)
            s = fallback_summarize(row['skills'])
    else:
        s = fallback_summarize(row['skills'])
    summaries.append(s)

df['summary'] = summaries
display(df[['candidate_id','name','skills','experience','summary']])


No OpenAI key found. Using fallback rule-based summarizer.


,candidate_id,name,skills,experience,summary
0,1,Rohit Das,"leadership,javascript,tableau,css",2,"Skills: leadership,javascript,tableau,css"
1,2,Nisha Ali,"terraform,docker,agile,kafka,kubernetes,produc...",10,"Skills: terraform,docker,agile,kafka,kubernete..."
2,3,Aishwarya Iyer,"aws,hadoop,scala,react,git",1,"Skills: aws,hadoop,scala,react,git"
3,4,Deepa Kumar,"flask,java,django,testing",9,"Skills: flask,java,django,testing"
4,5,Ajay Das,"javascript,gcp,analytics,powerbi",11,"Skills: javascript,gcp,analytics,powerbi"
...,...,...,...,...,...
115,116,Suman Sharma,"flask,tensorflow,analytics,pandas,ml",9,"Skills: flask,tensorflow,analytics,pandas,ml"
116,117,Raman Ali,"spring,leadership,python,pytorch",11,"Skills: spring,leadership,python,pytorch"
117,118,Monika Das,"aws,product-management,sql,kafka,xgboost",5,"Skills: aws,product-management,sql,kafka,xgboost"
118,119,Dipti Kumar,"leadership,airflow,scala,java,django,javascrip...",3,"Skills: leadership,airflow,scala,java,django,j..."


In [46]:
# Cell 6 — Feature engineering
# Create features: skill tokens, num_skills, experience, seniority
def extract_skills(skills_text):
    toks = [s.strip().lower() for s in skills_text.split(",") if s.strip()]
    return toks

df['skill_list'] = df['skills'].apply(extract_skills)
df['num_skills'] = df['skill_list'].apply(len)
df['seniority'] = pd.cut(df['experience'], bins=[-1,2,4,10], labels=['junior','mid','senior'])

# Create a skills vocabulary for vectorization (use TF-IDF on skills joined)
df['skills_joined'] = df['skill_list'].apply(lambda L: " ".join(L))
vect = TfidfVectorizer()
X_skills = vect.fit_transform(df['skills_joined'])

# numeric features matrix
X_num = df[['experience','num_skills']].values
from scipy.sparse import hstack
X = hstack([X_skills, X_num])
display(df[['candidate_id','name','skill_list','num_skills','experience','seniority']])


,candidate_id,name,skill_list,num_skills,experience,seniority
0,1,Rohit Das,"[leadership, javascript, tableau, css]",4,2,junior
1,2,Nisha Ali,"[terraform, docker, agile, kafka, kubernetes, ...",7,10,senior
2,3,Aishwarya Iyer,"[aws, hadoop, scala, react, git]",5,1,junior
3,4,Deepa Kumar,"[flask, java, django, testing]",4,9,senior
4,5,Ajay Das,"[javascript, gcp, analytics, powerbi]",4,11,NaN
...,...,...,...,...,...,...
115,116,Suman Sharma,"[flask, tensorflow, analytics, pandas, ml]",5,9,senior
116,117,Raman Ali,"[spring, leadership, python, pytorch]",4,11,NaN
117,118,Monika Das,"[aws, product-management, sql, kafka, xgboost]",5,5,senior
118,119,Dipti Kumar,"[leadership, airflow, scala, java, django, jav...",7,3,mid


In [47]:
# Cell 7 — Create labels (toy labels) and train/test split
# For demo: label = 1 (Good fit) if experience >=5 or 'ml' in skills OR 'xgboost' in skills else 0
def label_row(r):
    skills = " ".join(r['skill_list'])
    if r['experience'] >= 5 or 'ml' in skills or 'xgboost' in skills or 'deep-learning' in skills:
        return 1
    return 0

df['label_good_fit'] = df.apply(label_row, axis=1)
y = df['label_good_fit'].values
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, df.index, test_size=0.3, random_state=42)

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])


Train size: 84 Test size: 36


In [48]:
# Cell 8 — Train a RandomForest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train.toarray(), y_train)   # small dataset — convert to dense safely here
y_pred = clf.predict(X_test.toarray())

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save model & vectorizer
joblib.dump({"vect":vect, "clf":clf}, "colab_smart_hire_model.pkl")
print("Model saved to colab_smart_hire_model.pkl")


Accuracy: 0.9444444444444444
              precision    recall  f1-score   support

           0       0.71      1.00      0.83         5
           1       1.00      0.94      0.97        31

    accuracy                           0.94        36
   macro avg       0.86      0.97      0.90        36
weighted avg       0.96      0.94      0.95        36

Model saved to colab_smart_hire_model.pkl


In [49]:
# Cell 9 — Predictions on full dataset & simple ranking
X_full = X
preds = clf.predict(X_full.toarray())
df['pred_good_fit'] = preds
df['pred_prob'] = clf.predict_proba(X_full.toarray())[:,1]
df_sorted = df.sort_values('pred_prob', ascending=False)
display(df_sorted[['candidate_id','name','experience','skills','pred_good_fit','pred_prob']])


,candidate_id,name,experience,skills,pred_good_fit,pred_prob
20,21,Krishna Sharma,10,"airflow,devops,data-engineering,kafka,azure",1,1.00
54,55,Jaya Roy,7,"etl,terraform,xgboost,agile,leadership,linux,p...",1,1.00
38,39,Raman Roy,5,"terraform,hive,tensorflow,node",1,0.99
53,54,Vivaan Roy,5,"devops,python,tensorflow,pytorch,html",1,0.99
33,34,Raj Roy,12,"airflow,pytorch,tensorflow,react,product-manag...",1,0.99
...,...,...,...,...,...,...
50,51,Varun Das,2,"tensorflow,testing,data-engineering,react,numpy",0,0.10
48,49,Sanya Roy,2,"powerbi,product-management,java,scikit-learn,r...",0,0.10
2,3,Aishwarya Iyer,1,"aws,hadoop,scala,react,git",0,0.09
49,50,Parth Sharma,4,"agile,leadership,flask,pytorch,testing,scala",0,0.09


In [50]:
# Cell 10 — Quick dashboard: experience vs predicted probability & skill counts
fig = px.scatter(df, x='experience', y='pred_prob', size='num_skills',
                 hover_data=['name','skills'], title="Candidate predicted fit probability")
fig.show()

# Bar plot of top skills frequency
from collections import Counter
all_skills = [s for row in df['skill_list'] for s in row]
cnt = Counter(all_skills)
skill_df = pd.DataFrame(cnt.items(), columns=['skill','count']).sort_values('count', ascending=False)
fig2 = px.bar(skill_df, x='skill', y='count', title="Skill frequency")
fig2.show()


In [51]:
# Cell 11 — Download model artifact (optional)
from google.colab import files
files.download("colab_smart_hire_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [58]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK data (only need to run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added this line

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

**Sample Data (Resume & Job Descriptions)**


Here, we define our sample data.

resume_text: A sample applicant resume.

jd_text_1: A job description that should be a good match.

jd_text_2: A job description that should be a poor match.

jd_text_3: A job description that should be a medium match.

In [56]:
# Sample Resume
resume_text = """
John Doe
Senior Python Developer
Email: john.doe@email.com | LinkedIn: linkedin.com/in/johndoe

Summary:
Highly skilled Senior Python Developer with over 8 years of experience in backend development,
data analysis, and API design. Proficient in Django, Flask, and Pandas.
Experienced in leading development teams and mentoring junior engineers.
Strong background in data structures, algorithms, and cloud computing with AWS.

Experience:
Senior Python Developer | TechCorp Inc. | 2018 - Present
- Led the development of a high-traffic REST API using Django REST Framework.
- Optimized database queries, improving API response time by 30%.
- Integrated machine learning models into production services.
- Mentored a team of 4 junior developers.

Skills:
- Languages: Python, SQL
- Frameworks: Django, Flask, Pandas, NumPy
- Cloud: AWS (EC2, S3, Lambda)
- Databases: PostgreSQL, MySQL
- Other: Git, Docker, Agile
"""

# Sample Job Descriptions
jd_text_1 = """
Job Title: Senior Python Backend Developer
Company: DataSolutions LLC

Job Description:
We are seeking an experienced Senior Python Developer to join our backend team.
The ideal candidate will have strong experience with Python, Django, and Flask.
You will be responsible for designing and implementing robust REST APIs,
managing database interactions (PostgreSQL), and deploying applications on AWS.
Experience with data analysis libraries like Pandas is a major plus.
Must be a team player and have experience in an Agile environment.

Requirements:
- 5+ years of experience in Python development.
- Expertise in Django REST Framework and Flask.
- Strong knowledge of SQL and PostgreSQL.
- Experience with AWS cloud services.
- Familiarity with Git and Docker.
"""

jd_text_2 = """
Job Title: Front-End React Developer
Company: WebWidgets Co.

Job Description:
We are looking for a creative Front-End Developer to build user-facing applications.
Your primary role will be developing modern, responsive, and intuitive UIs using React.js.
You will work closely with our design team to translate mockups into high-quality code.

Requirements:
- 3+ years of experience with JavaScript and React.js.
- Strong proficiency in HTML, CSS, and modern web standards.
- Experience with state management tools like Redux or Context API.
- Familiarity with REST APIs (consuming them, not building).
- A keen eye for design and user experience.
"""

jd_text_3 = """
Job Title: Data Analyst
Company: Insights Inc.

Job Description:
We are hiring a Data Analyst to help us make sense of our data.
You will be responsible for cleaning, analyzing, and visualizing data to provide actionable insights.
Must be proficient in SQL and data visualization tools like Tableau or Power BI.
Some scripting experience in Python (with Pandas) is highly desirable for automation.

Requirements:
- 2+ years of experience as a Data Analyst.
- Expertise in SQL.
- Proficiency in Tableau or Power BI.
- Strong analytical and problem-solving skills.
- Python and Pandas experience is a plus.
"""

**Text Preprocessing**

A simple function to clean the text. This is crucial for TF-IDF, as it removes "noise" like punctuation and common "stop words" (e.g., 'the', 'is', 'a').

In [60]:
def preprocess_text(text):
    """Cleans and preprocesses text for TF-IDF."""
    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stop words
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word.isalpha()]

    return ' '.join(filtered_tokens)

# Create a list of all documents
documents = [resume_text, jd_text_1, jd_text_2, jd_text_3]
doc_names = ['Resume', 'JD 1 (Good Match)', 'JD 2 (Poor Match)', 'JD 3 (Medium Match)']

# Preprocess all documents
processed_docs = [preprocess_text(doc) for doc in documents]

# --- Optional: View processed text ---
# print(f"--- Original Resume --- \n{resume_text[:200]}...")
# print(f"\n--- Processed Resume --- \n{processed_docs[0][:200]}...")

**Core Algorithm: TF-IDF & Cosine Similarity**


This is the main logic.

We put all processed documents (resume + JDs) into a list.

The TfidfVectorizer learns the vocabulary and converts each document into a numerical vector.

We use cosine_similarity to compare the resume's vector (index 0) against the vectors for all the JDs (index 1:).

In [61]:
# 1. Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# 2. Fit and transform the documents
tfidf_matrix = vectorizer.fit_transform(processed_docs)

# 3. Calculate Cosine Similarity
# We compare the first document (Resume) to all other documents (JDs)
cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])

# 4. Display the Results
scores = cosine_sim[0]
job_matches = list(zip(doc_names[1:], scores))

# Sort by score in descending order
job_matches.sort(key=lambda x: x[1], reverse=True)

print("--- TF-IDF Matching Results ---")
for job, score in job_matches:
    print(f"{job}: {score:.4f}")

# Create a DataFrame for a cleaner look
df_tfidf = pd.DataFrame(job_matches, columns=['Job Description', 'TF-IDF Score'])
df_tfidf

--- TF-IDF Matching Results ---
JD 1 (Good Match): 0.4904
JD 3 (Medium Match): 0.1787
JD 2 (Poor Match): 0.1231


,Job Description,TF-IDF Score
0,JD 1 (Good Match),0.490363
1,JD 3 (Medium Match),0.178670
2,JD 2 (Poor Match),0.123077


**Matching Insights: Skill Overlap Analysis**

the score alone isn't enough. We can add a simple keyword-based overlap analysis to see why they matched.

In [62]:
def get_skill_overlap(resume_processed, jd_processed, skill_list):
    """Finds common skills between a resume and a JD."""
    resume_skills = set(word_tokenize(resume_processed))
    jd_skills = set(word_tokenize(jd_processed))

    # Find skills from our list that are in both
    common_skills = [skill for skill in skill_list if skill in resume_skills and skill in jd_skills]
    return common_skills

# Define a simple list of relevant skills (should be preprocessed)
# Note: These should be lowercase and stemmed/lemmatized in a real app
SKILL_LIST = [
    'python', 'django', 'flask', 'api', 'rest', 'data', 'analysis',
    'pandas', 'sql', 'postgresql', 'aws', 'docker', 'git', 'agile',
    'react', 'javascript', 'html', 'css', 'tableau'
]

print("--- Skill Overlap Analysis ---")

# Compare resume (processed_docs[0]) with each JD
for i, name in enumerate(doc_names[1:], 1):
    jd_doc = processed_docs[i]
    overlap = get_skill_overlap(processed_docs[0], jd_doc, SKILL_LIST)
    print(f"\n📄 Resume vs. {name}")
    print(f"Overlap: {', '.join(overlap) or 'None'}")

--- Skill Overlap Analysis ---

📄 Resume vs. JD 1 (Good Match)
Overlap: python, django, flask, rest, data, analysis, pandas, sql, postgresql, aws, docker, git, agile

📄 Resume vs. JD 2 (Poor Match)
Overlap: api, rest

📄 Resume vs. JD 3 (Medium Match)
Overlap: python, data, pandas, sql


**Advanced: Swap with BERT Embeddings**

TF-IDF is fast but "dumb"—it just counts words. BERT (using sentence-transformers) understands the semantic meaning of sentences. This can be much more powerful.

In [66]:
!pip install -U sentence-transformers

In [64]:
from sentence_transformers import SentenceTransformer, util

# 1. Load a pre-trained model
# 'all-MiniLM-L6-v2' is a great, fast model for this
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Generate embeddings
# IMPORTANT: BERT models work best on raw, UNPROCESSED text.
embeddings = model.encode(documents)

# 3. Calculate Cosine Similarity
# We use the PyTorch-based cosine similarity from the library
bert_cosine_sim = util.cos_sim(embeddings[0], embeddings[1:])

# 4. Display Results
bert_scores = bert_cosine_sim[0].cpu().numpy() # Move tensor to CPU and convert to numpy
bert_job_matches = list(zip(doc_names[1:], bert_scores))

# Sort by score in descending order
bert_job_matches.sort(key=lambda x: x[1], reverse=True)

print("\n--- BERT (Semantic) Matching Results ---")
for job, score in bert_job_matches:
    print(f"{job}: {score:.4f}")

# Create a DataFrame for comparison
df_bert = pd.DataFrame(bert_job_matches, columns=['Job Description', 'BERT Score'])
df_bert


--- BERT (Semantic) Matching Results ---
JD 1 (Good Match): 0.8290
JD 3 (Medium Match): 0.4899
JD 2 (Poor Match): 0.3927


,Job Description,BERT Score
0,JD 1 (Good Match),0.829044
1,JD 3 (Medium Match),0.489942
2,JD 2 (Poor Match),0.392650


In [65]:
# Combine for final comparison
df_final = pd.merge(df_tfidf, df_bert, on='Job Description')
print("--- Final Comparison ---")
print(df_final)

--- Final Comparison ---
       Job Description  TF-IDF Score  BERT Score
0    JD 1 (Good Match)      0.490363    0.829044
1  JD 3 (Medium Match)      0.178670    0.489942
2    JD 2 (Poor Match)      0.123077    0.392650
